In [1]:
import keras
import tensorflow

tensorflow.compat.v1.disable_eager_execution()

In [2]:
import gymnasium as gym

class GymnasiumToGymWrapper:
    def __init__(self, env):
        self.env = env
        self.action_space = env.action_space
        self.observation_space = env.observation_space

    def reset(self):
        obs, info = self.env.reset()
        return obs

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        done = terminated or truncated
        return obs, reward, done, info

    def render(self, mode="human"):
        return self.env.render()

    def close(self):
        return self.env.close()

In [3]:
import gymnasium as gym
import random
import pygame

env = gym.make("CartPole-v1", render_mode="human")
states=env.observation_space.shape[0]
actions=env.action_space.n

episodes=10

for episode in range (1, episodes+1):
    state = env.reset()
    done=False
    score=0

    while not done:
        env.render()
        action=random.choice([0, 1])
        n_state, reward, terminated, truncated, info = env.step(action)
        done=terminated or truncated
        score+=reward
    print(f'Episode:{episode}, Score:{score}')

print("states:", states)
print("actions:", actions)
env.close()

Episode:1, Score:17.0
Episode:2, Score:13.0
Episode:3, Score:23.0
Episode:4, Score:26.0
Episode:5, Score:16.0
Episode:6, Score:23.0
Episode:7, Score:16.0
Episode:8, Score:16.0
Episode:9, Score:33.0
Episode:10, Score:23.0
states: 4
actions: 2


In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten

def build_model(states, actions):
    model = Sequential()
    model.add(Flatten(input_shape=(1, states)))
    model.add(Dense(7, activation='swish'))
    model.add(Dense(15, activation='swish'))
    model.add(Dense(actions, activation='linear'))
    return model

model = build_model(states, actions)
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten (Flatten)           (None, 4)                 0         
                                                                 
 dense (Dense)               (None, 7)                 35        
                                                                 
 dense_1 (Dense)             (None, 15)                120       
                                                                 
 dense_2 (Dense)             (None, 2)                 32        
                                                                 
Total params: 187
Trainable params: 187
Non-trainable params: 0
_________________________________________________________________


In [4]:
import rl
from rl.agents import DQNAgent
from rl.policy import BoltzmannQPolicy
from rl.memory import SequentialMemory


def build_agent(model, actions):
    policy = BoltzmannQPolicy()
    
    memory = SequentialMemory(
        limit=50000,
        window_length=1
    )
    
    dqn = DQNAgent(
        model=model,
        memory=memory,
        policy=policy,
        nb_actions=actions,
        nb_steps_warmup=10,
        target_model_update=1e-2
    )
    
    return dqn

In [ ]:
from tensorflow.keras.optimizers import Adam
import gymnasium as gym
import numpy as np

env = GymnasiumToGymWrapper(gym.make("CartPole-v1"))

states = env.observation_space.shape[0]
actions = env.action_space.n

dqn = build_agent(model, actions)
dqn.compile(Adam(learning_rate=1e-3), metrics=["mae"])
dqn.fit(env, nb_steps=70000, visualize=False, verbose=1)

Training for 70000 steps ...
Interval 1 (0 steps performed)


D:\Anaconda\envs\TinyML_project\lib\site-packages\keras\engine\training_v1.py:2356: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,


    1/10000 [..............................] - ETA: 1:31:21 - reward: 1.0000

D:\Anaconda\envs\TinyML_project\lib\site-packages\rl\memory.py:37: UserWarning: Not enough entries to sample without replacement. Consider increasing your warm-up phase to avoid oversampling!
  warnings.warn('Not enough entries to sample without replacement. Consider increasing your warm-up phase to avoid oversampling!')


 3566/10000 [=========>....................] - ETA: 1:24 - reward: 1.0000

In [1]:
scores = dqn.test(env, nb_episodes=10, visualize=True)

print(np.mean(scores.history['episode_reward']))

NameError: name 'dqn' is not defined

In [ ]:
env = GymnasiumToGymWrapper(
    gym.make("CartPole-v1", render_mode="human")
)

_ = dqn.test(env, nb_episodes=10, visualize=True)

env.close()

In [5]:
dqn.save_weights("dqn_cartpole_weights.h5f", overwrite=True)

NameError: name 'dqn' is not defined

In [10]:
import gymnasium as gym
from tensorflow.keras.optimizers import Adam

env = GymnasiumToGymWrapper(gym.make("CartPole-v1", render_mode="human"))

states = env.observation_space.shape[0]
actions = env.action_space.n

model = build_model(states, actions)

dqn = build_agent(model, actions)
dqn.compile(Adam(learning_rate=1e-3), metrics=["mae"])

dqn.load_weights("dqn_cartpole_weights.h5f")

_ = dqn.test(env, nb_episodes=5, visualize=False)

Testing for 5 episodes ...
Episode 1: reward: 500.000, steps: 500
Episode 2: reward: 500.000, steps: 500
Episode 3: reward: 500.000, steps: 500
Episode 4: reward: 500.000, steps: 500
Episode 5: reward: 500.000, steps: 500
